# Оптимизация гиперпараметров
Датасет **Titanic-Dataset**: 
- целевая переменная - **Survived** 


In [3]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import kagglehub
from kagglehub import KaggleDatasetAdapter

%matplotlib inline
sns.set_theme(style="darkgrid")

file_path = "Titanic-Dataset.csv"

df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "yasserh/titanic-dataset",
  file_path,
)


pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

print(df.shape)
print(df.head())
print(df["Survived"].describe())

print("Количество пропусков в каждом столбце:")
display(df.isna().sum().to_frame(name="missing_values").T)
print(f"Количество полных дубликатов строк: {df.duplicated().sum()}")

c:\Users\danii\Desktop\MAI\ML\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(891, 12)
   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   NaN    

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
missing_values,0,0,0,0,0,177,0,0,0,0,687,2


Количество полных дубликатов строк: 0


**Объем и целостность данных:** Датасет содержит 891 объект и 12 столбцов (числовые: `Age`, `Fare`, `SibSp`, `Parch`; порядковые/бинарные: `Survived`, `Pclass`; строковые/категориальные: `Name`, `Sex`, `Ticket`, `Cabin`, `Embarked`). Полных дубликатов строк нет. Много пропусков номера кабины, меньше - возраст и порт посадки

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer

from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix


RANDOM_STATE = 42

def extract_deterministic_features(df_in):
    d = df_in.copy()
    
    d['Age_is_missing'] = d['Age'].isna().astype(int)
    d['Has_Cabin'] = d['Cabin'].notna().astype(int)
    
    cabin_map = {'A': 8, 'B': 7, 'C': 6, 'D': 5, 'E': 4, 'F': 3, 'G': 2, 'T': 1}
    d['Cabin'] = d['Cabin'].str[0].map(cabin_map).fillna(0).astype(int)
    
    d['Sex'] = d['Sex'].map({'male': 1, 'female': 0})
    d['Pclass'] = d['Pclass'].map({1: 3, 2: 2, 3: 1})
    return d

class GroupMedianAgeImputer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.group_medians_ = X.groupby(['Pclass', 'Sex'])['Age'].median()
        self.global_median_ = X['Age'].median()
        return self

    def transform(self, X):
        X_out = X.copy()
        mapped_age = (
            X_out.set_index(['Pclass', 'Sex'])
            .index.map(self.group_medians_)
            .to_series(index=X_out.index)
        )
        X_out['Age'] = X_out['Age'].fillna(mapped_age).fillna(self.global_median_)
        return X_out

numeric_features = ['Age', 'Fare', 'SibSp', 'Parch', 'Pclass', 'Cabin', 'Sex', 'Age_is_missing']
categorical_features = ['Embarked']

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

column_preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ],
    remainder='drop'
)

preprocessor = Pipeline(steps=[
    ('feature_extraction', FunctionTransformer(extract_deterministic_features)),
    ('age_group_imputer', GroupMedianAgeImputer()),
    ('column_processing', column_preprocessor)
])

### Предобработка данных:
Первичный анализ данных пропущен, т.к. был произведён в прошлой ЛР. Полученные с прошлой ЛР данные будут переиспользованы для заполнения пропусков. Ключевые моменты:

- **Обработка пропусков и категорий:**
   * `Embarked` (2 пропуска): заполняется модой (`most_frequent`) и кодируется через `OneHotEncoder` в 3 бинарных признака, так как между портами посадки нельзя определить, какой больше влияет на выживаемость.
   * `Cabin` (687 пропусков): извлекается первая буква палубы и переводится в порядковую шкалу от `1` (нижняя палуба `T`) до `8` (верхняя палуба `A`), а пропуски кодируются как `0`.
   * `Age` (177 пропусков): заполняется медианным возрастом внутри групп `['Pclass', 'Sex']` (с помощью кастомного трансформера `GroupMedianAgeImputer`), так как возраст заметно коррелирует с классом билета (0.41). Дополнительно сохраняется бинарный индикатор пропуска `Age_is_missing`.


In [12]:
from sklearn.model_selection import GridSearchCV, cross_val_predict


X = df.drop(columns=["Survived"])
y = df["Survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE
)

pipe = Pipeline([ 
    ('preprocessor', preprocessor),
    ('classifier', GradientBoostingClassifier(random_state=RANDOM_STATE))
])

param_grid = {
    'classifier__n_estimators': [100, 150, 250, 500, 700, 1000],
    'classifier__learning_rate': [0.05, 0.075, 0.1, 0.15, 0.2],
    'classifier__max_depth': [2, 3, 4, 5, 7, 8, 9, 10],
    'classifier__subsample': [0.6, 0.8, 0.9, 0.95, 1.0]
}


skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

grid_search = GridSearchCV(
    pipe,               
    param_grid=param_grid,
    cv=skf,
    scoring='f1',
    n_jobs=-1
)
grid_search.fit(X_train, y_train)

best_comp_pipe = grid_search.best_estimator_ #готовая модель, обученная на лучших параметрах

print(f"Лучшие параметры: {grid_search.best_params_}")
print(f"F1 на 5-fold CV (при стандартном пороге 0.50): {grid_search.best_score_:.4f}")

best_idx = grid_search.best_index_
cv_results = grid_search.cv_results_

fold_f1_scores = [
    cv_results[f'split{i}_test_score'][best_idx]
    for i in range(grid_search.n_splits_)
]

print("F1 по фолдам:")
for i, score in enumerate(fold_f1_scores, start=1):
    print(f"\tFold {i}: {score:.4f}")

Лучшие параметры: {'classifier__learning_rate': 0.075, 'classifier__max_depth': 3, 'classifier__n_estimators': 150, 'classifier__subsample': 0.6}
F1 на 5-fold CV (при стандартном пороге 0.50): 0.7760
F1 по фолдам:
	Fold 1: 0.8037
	Fold 2: 0.7339
	Fold 3: 0.8000
	Fold 4: 0.7500
	Fold 5: 0.7921
